Requires **flowmeter** — install once from the repo root:

```bash
pip install -e tools/flowmeter/
```

# Imports

In [ ]:
import os
import glob
import tempfile

import pandas as pd
from flowmeter import convert_pcap_to_csv

# Inputs

In [ ]:
FOLDERS_PATHS = [
    "../data/raw/CIC_IIoT_dataset_2025/PCAPs/benign",
    "../data/raw/CIC_IIoT_dataset_2025/PCAPs/bruteforce",
    "../data/raw/CIC_IIoT_dataset_2025/PCAPs/ddos",
    "../data/raw/CIC_IIoT_dataset_2025/PCAPs/dos",
    "../data/raw/CIC_IIoT_dataset_2025/PCAPs/malware",
    "../data/raw/CIC_IIoT_dataset_2025/PCAPs/mitm",
    "../data/raw/CIC_IIoT_dataset_2025/PCAPs/recon",
    "../data/raw/CIC_IIoT_dataset_2025/PCAPs/web"
]

OUTPUT_DIR = "../data/raw/CIC_IIoT_dataset_2025/generated_CSVs"

# Preprocessing

In [ ]:
def generate_merged_csv_from_pcaps(input_pcap_dir, output_csv_dir, output_filename="merged_output.csv"):
    """
    Convert every PCAP/pcapng in a folder to flow features and merge into one CSV.

    Parameters
    ----------
    input_pcap_dir : str
        Folder containing .pcap / .pcapng files.
    output_csv_dir : str
        Destination folder for the merged CSV.
    output_filename : str
        Name of the merged output file (default: merged_output.csv).

    Returns
    -------
    str
        Full path to the merged CSV file.
    """
    if not os.path.isdir(input_pcap_dir):
        raise FileNotFoundError(f"Input folder does not exist: {input_pcap_dir}")

    os.makedirs(output_csv_dir, exist_ok=True)

    pcap_files = sorted(
        glob.glob(os.path.join(input_pcap_dir, "*.pcap")) +
        glob.glob(os.path.join(input_pcap_dir, "*.pcapng"))
    )

    if not pcap_files:
        raise FileNotFoundError(f"No PCAP files found in: {input_pcap_dir}")

    print(f"Found {len(pcap_files)} PCAP file(s) in {input_pcap_dir}")

    frames = []
    with tempfile.TemporaryDirectory() as tmp_dir:
        for i, pcap_path in enumerate(pcap_files, 1):
            pcap_name = os.path.basename(pcap_path)
            print(f"[{i}/{len(pcap_files)}] Processing {pcap_name}...")

            tmp_csv = os.path.join(tmp_dir, f"{pcap_name}.csv")
            n_flows = convert_pcap_to_csv(pcap_path, tmp_csv)
            print(f"  → {n_flows} flows extracted")

            if n_flows > 0:
                frames.append(pd.read_csv(tmp_csv))

    if not frames:
        raise RuntimeError("No flows were extracted from any PCAP file.")

    merged_csv_path = os.path.join(output_csv_dir, output_filename)
    merged_df = pd.concat(frames, ignore_index=True)
    merged_df.to_csv(merged_csv_path, index=False)

    print(f"\nMerged CSV: {merged_csv_path}  ({len(merged_df)} flows total)")
    return merged_csv_path

In [ ]:
for folder_path in FOLDERS_PATHS:
    generate_merged_csv_from_pcaps(folder_path, OUTPUT_DIR)